# Strategy Research Walkthrough

Build, backtest, and analyze a momentum strategy from scratch using `quant.engine`.

In [19]:
import sys
sys.path.append("..")
import os
import pandas as pd
import numpy as np
from engine import Strategy, Signal, Engine, BacktestConfig, DataFrameSource, summary
from engine.report import generate

# Set GCS bucket for loading backfilled data
os.environ["QUANT_GCS_BUCKET"] = "deductive-notch-495015-c2-quant-data"

print("Engine ready.")

Engine ready.


## 1. Load Real Data from GCS

Works once the backfill job completes. For now, use the recent data that's already in GCS.

In [20]:
from quant.direct import bars_direct

# Load S&P 500 stocks from GCS backfill (QQQ/IWM/XLE are ETFs, not in S&P 500)
df = bars_direct(
    ["NVDA", "MSFT", "AAPL", "AMZN"],
    start="2025-01-01",
    end="2025-05-31",
    market="us",
)
print(f"Shape: {df.shape}")
print(f"Symbols: {df.index.get_level_values('symbol').unique().tolist()}")

# Convert MultiIndex (symbol, timestamp) to pivot table for the engine
close = df["close"].unstack("symbol")  # columns=symbols, index=timestamp
print(f"Close shape: {close.shape}")
close.head()

Shape: (408, 7)
Symbols: ['AAPL', 'AMZN', 'MSFT', 'NVDA']
Close shape: (102, 4)


symbol,AAPL,AMZN,MSFT,NVDA
timestamp,,,,
2025-01-02 00:00:00+00:00,242.301926,220.220001,414.568604,138.264694
2025-01-03 00:00:00+00:00,241.815033,224.190002,419.292877,144.422684
2025-01-06 00:00:00+00:00,243.444595,227.610001,423.749756,149.381042
2025-01-07 00:00:00+00:00,240.672348,222.110001,418.322266,140.094086
2025-01-08 00:00:00+00:00,241.159225,222.130005,420.491302,140.064102


## 2. Define a Momentum Strategy

Buy top-N momentum stocks, rebalance monthly.

In [ ]:
class MomentumStrategy(Strategy):
    lookback: int = 60        # 60-day momentum
    top_n: int = 2            # Hold top 2 stocks
    rebalance_days: int = 21  # Rebalance every 21 bars (~monthly)

    def on_init(self, ctx):
        self.bars_since_rebalance = 0

    def on_bar(self, ctx, bar):
        self.bars_since_rebalance += 1
        # Not enough history yet — wait for lookback window
        if bar < self.lookback:
            return []
        if self.bars_since_rebalance < self.rebalance_days:
            return []

        # Compute momentum for each symbol as a Series
        momentum = (ctx.data.close.iloc[bar] / ctx.data.close.iloc[bar - self.lookback]) - 1
        if hasattr(momentum, "nlargest"):
            top = momentum.nlargest(self.top_n).index.tolist()
        else:
            top = [ctx.universe[0]]  # Single-symbol fallback

        signals = []
        for sym in ctx.universe:
            if ctx.portfolio.has_position(sym) and sym not in top:
                signals.append(Signal.close(sym))
        for sym in top:
            if not ctx.portfolio.has_position(sym):
                signals.append(Signal.target(sym, weight=1.0 / self.top_n))

        self.bars_since_rebalance = 0
        return signals

print(f"Parameters: {MomentumStrategy().parameters()}")

## 3. Run Backtest

In [22]:
from engine.data import DataFrameSource

data = DataFrameSource(close=close)
cfg = BacktestConfig(initial_capital=100_000, slippage_bps=5, commission_bps=1)
engine = Engine(cfg)
result = engine.run(MomentumStrategy(), data)

s = summary(result)
print(f"Strategy: {result.strategy_name}")
print(f"Total Return: {s['total_return']:.2%}")
print(f"Annual Return: {s['annual_return']:.2%}")
print(f"Sharpe Ratio: {s['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {s['max_drawdown']:.2%}")
print(f"Win Rate: {s['win_rate']:.1%}")

Strategy: MomentumStrategy
Total Return: -362.15%
Annual Return: nan%
Sharpe Ratio: -0.66
Max Drawdown: -339.98%
Win Rate: 55.5%


d:\quant\notebooks\..\engine\metrics.py:12: RuntimeWarning: invalid value encountered in scalar power
  ann_ret = (1 + total_ret) ** (1 / n_years) - 1


## 4. Visualize Equity Curve

In [23]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

eq = result.portfolio.equity_curve
ax1.plot(eq.index, eq.values, color="#1a1a2e", linewidth=1)
ax1.set_title("Equity Curve")
ax1.set_ylabel("Portfolio Value ($)")
ax1.grid(True, alpha=0.3)

rolling_max = eq.expanding().max()
dd = (eq - rolling_max) / rolling_max
ax2.fill_between(dd.index, dd.values, 0, color="#e74c3c", alpha=0.3)
ax2.plot(dd.index, dd.values, color="#e74c3c", linewidth=0.5)
ax2.set_title("Drawdown")
ax2.set_ylabel("Drawdown %")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\DangXuan\AppData\Local\Temp\ipykernel_22128\1839397408.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Analyze Performance by Month

In [24]:
monthly = eq.resample("ME").last().pct_change().dropna()
print(f"Positive months: {(monthly > 0).mean():.1%}")
print(f"Best month: {monthly.max():.2%}")
print(f"Worst month: {monthly.min():.2%}")

monthly.plot(kind="bar", figsize=(12, 3),
             color=["#2ecc71" if v >= 0 else "#e74c3c" for v in monthly])
plt.title("Monthly Returns")
plt.axhline(y=0, color="black", linewidth=0.5)
plt.show()

Positive months: 100.0%
Best month: 0.73%
Worst month: 0.11%


C:\Users\DangXuan\AppData\Local\Temp\ipykernel_22128\1017247451.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Iterate: Try Different Parameters

In [28]:
results = []
for lookback in [20, 60, 120]:
    for top_n in [1, 2, 3]:
        s = MomentumStrategy()
        s.lookback = lookback
        s.top_n = top_n
        r = Engine(cfg).run(s, data)
        m = summary(r)
        results.append({"lookback": lookback, "top_n": top_n,
                        "sharpe": m["sharpe_ratio"],
                        "return": m["annual_return"],
                        "max_dd": m["max_drawdown"]})

df_results = pd.DataFrame(results).sort_values("sharpe", ascending=False)
df_results.style.background_gradient(subset=["sharpe", "return", "max_dd"])

d:\quant\notebooks\..\engine\metrics.py:12: RuntimeWarning: invalid value encountered in scalar power
  ann_ret = (1 + total_ret) ** (1 / n_years) - 1


IndexError: single positional indexer is out-of-bounds

## 7. Generate Report

In [29]:
generate(result, "momentum_strategy_report.html")
print("Report saved to momentum_strategy_report.html")

Report saved to momentum_strategy_report.html


## Next Steps

- Load more symbols: `bars_direct(["SPY","AAPL","MSFT","NVDA","GOOGL"], start, end, market="us")`
- Add risk rules: `self.add_risk(StopLoss(pct=0.05))` in `on_init()`
- Run walk-forward: `WalkForward(strategy, data, cfg).summary()`
- Optimize with `GridSearch(strategy_class, param_grid, data, cfg)`
- View the full system in `docs/manual/02-research.md`